# Map of Italian Science — Country & Organisation Citation Analysis

## Overview

This notebook investigates the **structural composition** of international citation networks for six Italian research universities: UNIBO, UNIMI, UNIPD, UNITO, UPO, and SNS. The central question is *how those citations are distributed* across partner countries and their institutions.

A country with 50,000 citations could represent two entirely different realities: three flagship universities accounting for 90% of those citations — a fragile, hub-and-spoke structure — or hundreds of institutions each contributing a small share, indicating a resilient and systemically integrated network. Distinguishing between these two cases is the analytical task of this notebook.

The analysis moves through two scopes:

1. **Part I — Aggregate analysis** — a static snapshot of concentration structure across all years combined, establishing the baseline structural archetypes.
2. **Part II — Temporal analysis** — how the structure evolves over time, from early bilateral ties toward systemic integration.
3. **Part III — Structural analysis** — address the distribution question

---

## Research Questions

**RQ1 — Citation Concentration:**
For each partner country, what percentage of its total citations to a given Italian institution is generated by its top-3 organisations?

**RQ2 — Organisational Composition:**
Which specific organisations within each partner country drive the citation relationship, and how does their contribution compare across directions?

---

## Metrics

### Concentration Ratio — CR₃

The primary metric is the Concentration Ratio at N=3, which measures what fraction of a country's total citations to a given Italian institution come from that country's top-3 most-cited partner organisations:

$$CR_3(c) = \frac{\sum_{k=1}^{3} \text{citations from top-}k\text{ org in country }c}{\text{total citations from country }c} \times 100$$

- **CR₃ = 100%** → all citations come from a single organisation (maximum concentration)
- **CR₃ → 0%** → citations spread evenly across many organisations (maximum fragmentation)

---

## Data

**Institutions:** UNIBO · UNIMI · UNIPD · UNITO · UPO · SNS

**Files per institution (aggregate):** `citation_counts_organizations_incoming_clean.csv` / `citation_counts_organizations_outgoing_clean.csv`

**Files per institution (temporal):** same files disaggregated into 5-year blocks under `citation_counts_annualized/`

**Key columns:** `country_name`, `country_code`, `legal_name`, `count`, `institution`, `direction`

---

## Notebook Structure

> **Part I — Aggregate Concentration Analysis**
> - 1.1 Setup & data loading
> - 1.2 CR₃ calculation engine
> - 1.3 Scatter plot: citation volume vs. concentration
> - 1.4 Findings
>
> **Part II — Temporal Concentration Analysis (five 5-year blocks)**
> - 2.1 Setup & temporal data loading
> - 2.2 Scatter plot: how the volume–concentration relationship evolves
> - 2.3 CR₃ Snapshot Comparison: 2001–2005 vs 2021–2025
> - 2.4 Findings — 25-year network evolution
>
> **Part III - Citation structure**
> - 3.1 Sunburst: country and organisation citation structure
> - 3.2 Findings — organisational composition patterns


---
# Part I — Aggregate Concentration Analysis

## 1.1 Setup & Data Loading

The block below imports libraries and defines the two path constants that the rest of Part I depends on. `BASE_PATH` points to the **aggregate** organisation-level CSVs (all years combined). The six institution codes map to their full names via `INST_LABELS`, which every plot uses for readable titles.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output


# Paths & Environment
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "data_viz":
    sys.path.append(str(CURRENT_DIR))
else:
    sys.path.append(str(CURRENT_DIR))

# Project imports
from src.data_utils import *
from src.validation import *

# Visualization settings
INST_COLORS = {
    "UNIBO": "#264653", "UNIMI": "#2a9d8f", "UNIPD": "#8ab17d",
    "UNITO": "#e9c46a", "UPO":   "#f4a261", "SNS":   "#e76f51",
}
DIR_COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}
INST_LABELS = INSTITUTION_LABELS
DIR_LABELS  = {"incoming": "Incoming citations", "outgoing": "Outgoing citations"}


## 1.2 CR_N Calculation Engine

`calculate_concentration_metrics()` takes the combined master dataframe, a citation direction (`incoming` / `outgoing`), and a tier N, then returns one row per (institution × country) with:

- `total_country_citations` — the country's aggregate citation volume
- `top_n_citations` — sum of citations from the top-N organisations in that country  
- `CR_N` — the concentration ratio as a percentage

The loading function `load_and_calculate_concentration()` reads all available CSVs across the six institutions, combines them, and computes CR₃ for both directions. The function is tolerant of missing files (it silently skips them), so the notebook runs even on partial datasets.


In [2]:
# CR₃ calculation (aggregate)
def calculate_concentration_metrics(df_orgs, df_countries, direction='incoming', n=3):
    df_orgs_filtered = df_orgs[df_orgs['direction'] == direction].copy()
    if df_orgs_filtered.empty:
        return pd.DataFrame()

    weight_col = (
        'counts' if 'counts' in df_orgs_filtered.columns else
        'count'  if 'count'  in df_orgs_filtered.columns else
        'derived_counts'
    )
    if weight_col == 'derived_counts':
        df_orgs_filtered['derived_counts'] = 1

    df_countries_filtered = df_countries[df_countries['direction'] == direction].copy()
    country_weight_col = 'counts' if 'counts' in df_countries_filtered.columns else 'count'
    
    country_totals = (
        df_countries_filtered
        .groupby(['institution', 'country_name'])[country_weight_col]
        .sum().reset_index(name='total_country_citations')
    )
    country_totals = country_totals.rename(columns={'institution': 'italian_institution'})

    org_totals = (
        df_orgs_filtered
        .groupby(['italian_institution', 'country_name', 'legal_name'])[weight_col]
        .sum().reset_index(name='org_citations')
    )
    org_totals = org_totals.sort_values(
        ['italian_institution', 'country_name', 'org_citations'],
        ascending=[True, True, False]
    )
    top_n_orgs = org_totals.groupby(['italian_institution', 'country_name']).head(n)
    top_n_sums = (
        top_n_orgs
        .groupby(['italian_institution', 'country_name'])['org_citations']
        .sum().reset_index(name='top_n_citations')
    )
    cr_df = pd.merge(country_totals, top_n_sums,
                     on=['italian_institution', 'country_name'], how='left')
    cr_df['top_n_citations'] = cr_df['top_n_citations'].fillna(0)
    cr_df[f'CR_{n}'] = (
        cr_df['top_n_citations'] / cr_df['total_country_citations']
    ) * 100

    return cr_df.sort_values(
        by=['italian_institution', 'total_country_citations'],
        ascending=[True, False]
    )

viz_base_path = AGGREGATE_VISUALIZATIONS_PATH
datasets = load_all_available(base_path=viz_base_path, suffix='_clean')
country_datasets = load_all(base_path=viz_base_path, suffix='_clean')

if datasets and not country_datasets.empty:
    agg_master = pd.concat(
        [
            df.assign(italian_institution=inst)
            for inst, (inc, out) in datasets.items()
            for df in (inc, out)
        ],
        ignore_index=True,
    )
    static_incoming_cr_df = calculate_concentration_metrics(agg_master, country_datasets, 'incoming', 3)
    static_outgoing_cr_df = calculate_concentration_metrics(agg_master, country_datasets, 'outgoing', 3)
else:
    static_incoming_cr_df = None
    static_outgoing_cr_df = None

agg_incoming_cr_df = static_incoming_cr_df
agg_outgoing_cr_df = static_outgoing_cr_df


✓ UNIBO: incoming 52,384 orgs · outgoing 47,472 orgs
✓ UNIMI: incoming 53,092 orgs · outgoing 46,522 orgs
✓ UNIPD: incoming 51,536 orgs · outgoing 46,735 orgs
✓ UNITO: incoming 47,907 orgs · outgoing 43,608 orgs
✓ UPO: incoming 33,109 orgs · outgoing 30,151 orgs
✓ SNS: incoming 21,060 orgs · outgoing 15,765 orgs


## 1.3 Scatter plot: Citation Volume vs. Concentration

`plot_single_inst` creates two bubble scatter plots per selected institution — one for incoming citations (papers that cite this institution), one for outgoing (papers this institution cites). Each bubble is a partner country. Position encodes the core tension of this analysis: the x-axis shows how much that country contributes in raw citations (log scale), while the y-axis shows how concentrated those citations are in its top-3 organisations.

**How to read it:**

| Quadrant | What it means |
|---|---|
| Top-left: high CR_N, low volume | Niche or emerging partner — one or two foreign flagships drive most of the relationship |
| Top-right: high CR_N, high volume | Large science system but structurally dependent on a small elite |
| Bottom-left: low CR_N, low volume | Peripheral partner with broad but thin engagement |
| Bottom-right: low CR_N, high volume | Mature, deeply integrated partner — the ideal profile |

The **dashed line at CR_N = 50%** marks the "concentrated" threshold: above it, three organisations account for the majority of a country's citations. Use the dropdown to switch between institutions.

In [3]:
# Build a persistent color map
def _build_country_color_map(*cr_dfs):
    all_countries = sorted(set(
        country
        for df in cr_dfs if df is not None
        for country in df['country_name'].unique()
    ))
    palette = px.colors.qualitative.Pastel + px.colors.qualitative.Set3
    return {country: palette[i % len(palette)] for i, country in enumerate(all_countries)}

COUNTRY_COLOR_MAP = _build_country_color_map(static_incoming_cr_df, static_outgoing_cr_df)

def plot_single_inst(inst_name, cr_df, direction, n=3):
    plot_df = (
        cr_df[cr_df['italian_institution'] == inst_name]
        .sort_values('total_country_citations', ascending=False)
        .head(20)
        .copy() 
    )
    
    label      = INST_LABELS.get(inst_name, inst_name)
    base_color = DIR_COLORS[direction]
    badge_text = '▼ Incoming' if direction == 'incoming' else '▲ Outgoing'

    fig = px.scatter(
        plot_df,
        x='total_country_citations',
        y=f'CR_{n}',
        size='total_country_citations',
        color='country_name',
        text='country_name',
        log_x=True,
        template='plotly_white',
        height=550,
        color_discrete_map=COUNTRY_COLOR_MAP,
        labels={
            'total_country_citations': 'Total citations (log scale)',
            f'CR_{n}': f'CR{n} — top-{n} org concentration (%)',
            'country_name': 'Country',
        },
        title=(
            f'<b>Citation Concentration vs. Volume</b>  ·  {label}'
            f'  ·  {DIR_LABELS[direction]}'
        ),
    )
    fig.update_traces(textposition='top center', marker=dict(opacity=0.82))

    # Direction badge
    fig.add_annotation(
        xref='paper', yref='paper', x=0.0, y=1.07,
        text=(
            f'<span style="background:{base_color};color:white;'
            f'padding:2px 8px;border-radius:4px;font-size:12px">'
            f'{badge_text}</span>'
        ),
        showarrow=False, xanchor='left',
    )

    fig.add_hline(y=50, line_dash='dash', line_color=base_color, opacity=0.5)
    fig.add_annotation(
        text='50% concentration threshold',
        xref='paper', x=1.01, yref='y', y=50,
        xanchor='left', showarrow=False,
        font=dict(size=11, color=base_color),
    )
    fig.update_yaxes(range=[0, 115], title=f'CR{n} (%)')
    fig.update_layout(showlegend=True, margin=dict(r=160))
    return fig

if static_incoming_cr_df is not None:
    out_static = widgets.Output()

    inst_static_dd = widgets.Dropdown(
        options=[(INST_LABELS[i], i) for i in INSTITUTIONS],
        value='UNIBO',
        description='Institution:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px'),
    )
    dir_static_dd = widgets.Dropdown(
        options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
        value='incoming',
        description='Direction:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='230px'),
    )
    static_controls = widgets.HBox(
        [inst_static_dd, dir_static_dd],
        layout=widgets.Layout(gap='20px', margin='0 0 12px 0'),
    )

    def render_static(inst_name, direction):
        cr_df = static_incoming_cr_df if direction == 'incoming' else static_outgoing_cr_df
        with out_static:
            clear_output(wait=True)
            plot_single_inst(inst_name, cr_df, direction).show()

    def on_static_change(change):
        render_static(inst_static_dd.value, dir_static_dd.value)

    inst_static_dd.observe(on_static_change, names='value')
    dir_static_dd.observe(on_static_change, names='value')

    display(static_controls, out_static)
    render_static('UNIBO', 'incoming')

else:
    print("No data loaded — check BASE_PATH and file names.")

Output()

## 1.4 Findings

The scatter plots reveal a remarkably consistent structural picture across all six institutions and both citation directions. The core geometry is the same in every view: a dense cluster of major partners in the bottom-right (high volume, low concentration), a dispersed scatter of smaller partners in the left half of the chart, and an almost empty upper half of the plot. This last point is the most important finding: **with very few exceptions, no country in the top-20 by volume exceeds the 50% concentration threshold**. The concentrated zone is essentially vacant for all Italian institutions; the threshold becomes relevant only for UPO and SNS, which operate at smaller scale.

---

### The concentrated zone is almost empty

Across UNIBO, UNIMI, UNIPD, and UNITO — in both directions — not a single country in the top-20 by citation volume crosses or even approaches the 50% line. Every major partner sits well below 40% CR₃, and most of the high-volume cluster sits below 20%. This is not a mild version of fragmentation: it is near-complete absence of elite capture for the four largest institutions. For these universities, the question of "which three organisations dominate a country's citations" is analytically moot for every partner that matters by volume.

The picture changes only at UPO and SNS, where the smaller overall citation base means some countries appear above the threshold:

- **Finland** is the only country that consistently exceeds 50% CR₃ across multiple institutions and directions — visible in UPO incoming (~60%), UPO outgoing (~60%), SNS outgoing (~62%), and SNS incoming (~63%). At these smaller institutions, Finnish citations are effectively controlled by one or two organisations, almost certainly a single leading research university with a specific disciplinary link.
- **Switzerland** sits near or just above the 50% line in both SNS directions (~52–53%), consistent with concentrated elite engagement — likely the ETH domain — that has not yet diversified into the broader Swiss science system.
- **Belgium** (~52%) and **Hungary** (outgoing only) appear above the threshold exclusively at SNS, reflecting the institution's narrow disciplinary scope generating highly concentrated relationships with specific foreign partners.

The implication is clear: concentration risk is not a system-wide problem for Italian international citation networks. It is institution-specific and scale-dependent, concentrated at the two smallest institutions.


### The fragmented core: US, France, Germany, UK

These four countries occupy the extreme bottom-right in every single view. Their CR₃ values stay consistently below 15–20% regardless of institution or direction. Notably, this holds even for UPO and SNS despite their smaller scale — the structural fragmentation of engagement with the US, France, Germany, and the UK is a property of those countries' science systems, not of the Italian institution's size or profile.

The US is always the rightmost bubble by a substantial margin, with CR₃ near or below 5% everywhere. At UNIBO and UNIMI it reaches 2–4M citations while maintaining this near-zero concentration. This means that even taken together, the top-3 American institutions account for fewer than 5% of all US citations to these universities — a level of distribution that is essentially immune to any single institutional fluctuation.


### Italy's structural anomaly

Italy is the most analytically interesting outlier in the dataset. It consistently appears as a mid-field bubble — high volume but with CR₃ notably above the trend line for its size, typically in the 25–35% range across all institutions and both directions. This places it structurally closer to mid-tier partners like Switzerland or Brazil than to the major international science systems it sits alongside in terms of raw volume.

This is not an international dependency — it is a domestic structural feature. Cross-institutional citations within Italy are concentrated in the same few leading universities that appear at the top of Italian research output rankings. The Italian academic system generates a specific citation pattern: high volume but concentrated through a small number of research universities, which is precisely what a high-but-not-extreme CR₃ captures.


### Incoming vs. outgoing: structural symmetry with directional nuance

The most striking feature when switching directions is how similar the charts are. Country rankings, CR₃ values, and the overall geometry are preserved in almost every case. The main consistent difference is **volume**: outgoing citation counts are systematically lower than incoming across all institutions, which shifts the x-axis scale leftward. This reflects the general pattern that these Italian institutions receive more international citations than they generate — their output is internationally cited by a broad base, while their own reference lists, though broad, are somewhat more selective.

Beyond the volume shift, a few direction-specific differences are visible:

**SNS** shows the strongest directional divergence. In the outgoing chart, Spain sits at ~35% CR₃ — noticeably higher than its ~25% in the incoming view — and Hungary appears in the top-20 outgoing but not incoming. This is consistent with SNS's highly specialised disciplinary profile: its researchers cite a narrower, more concentrated set of foreign partners than the international community that cites SNS output. The outgoing network is more selective; the incoming network is broader.

**UNITO incoming** is the only chart where Taiwan appears in the top-20, suggesting a specific disciplinary import relationship (likely physics or mathematics) that does not manifest symmetrically in outgoing citations.

**UNIMI incoming** shows Sweden and Denmark near the threshold (~47–50%), while these countries sit lower in the outgoing view. UNIMI's incoming citations from Scandinavian countries appear to be driven by a handful of institutions, possibly in the life sciences.

**UPO** shows the highest directional symmetry of all six institutions — the charts are nearly identical in both directions, which is consistent with UPO being a smaller, more specialised university whose citation relationships are stable and bidirectional.

---

### Structural implications

The aggregate snapshot delivers three clear conclusions. First, concentration risk is effectively absent for the four largest institutions — the relevant policy question for UNIBO, UNIMI, UNIPD, and UNITO is not "are we over-reliant on a few foreign organisations?" but "how broadly are our international relationships distributed and growing?". Second, for UPO and SNS, Finland and Switzerland represent the only genuine structural concentrations worth monitoring, though even these may reflect disciplinary alignment rather than dependency. Third, Italy's anomalous mid-field position is a persistent domestic structural feature that deserves separate treatment in any analysis of citation network resilience.

---
# Part II — Temporal Concentration Analysis (five 5-year blocks)

This section adds the time dimension. Instead of a single aggregate snapshot, citation data is disaggregated into five consecutive 5-year blocks: 2001–2005, 2006–2010, 2011–2015, 2016–2020, and 2021–2025. The question shifts from *what is the structure?* to *how does the structure evolve?*

## 2.1 Setup & Temporal Data Loading

`ANNUALIZED_PATH` points to the temporal CSV files. `YEAR_BLOCKS` defines the five periods. All other constants (`INSTITUTIONS`, `INST_LABELS`, `DIR_COLORS`) are inherited from the Part I setup cell. The loader returns `master_df` alongside the pre-computed CR₃ dataframes — `master_df` is kept as a global because the HHI calculation in Section 2.4 operates directly on the raw organisation-level data.

In [4]:
# Part II configuration
YEAR_BLOCKS = ['2001-2005', '2006-2010', '2011-2015', '2016-2020', '2021-2025']


# Temporal CR₃ calculation
def calculate_concentration_metrics_temporal(df, direction='incoming', n=3):
    df_filtered = df[df['direction'] == direction].copy()
    if df_filtered.empty:
        return pd.DataFrame()

    weight_col = (
        'counts' if 'counts' in df_filtered.columns else
        'count'  if 'count'  in df_filtered.columns else
        'derived_counts'
    )
    if weight_col == 'derived_counts':
        df_filtered['derived_counts'] = 1

    country_totals = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name'])[weight_col]
        .sum().reset_index(name='total_country_citations')
    )
    
    org_totals = (
        df_filtered
        .groupby(['italian_institution', 'year_block', 'country_name', 'legal_name'])[weight_col]
        .sum().reset_index(name='org_citations')
    )
    org_totals = org_totals.sort_values(
        ['italian_institution', 'year_block', 'country_name', 'org_citations'],
        ascending=[True, True, True, False]
    )
    top_n_orgs = org_totals.groupby(
        ['italian_institution', 'year_block', 'country_name']
    ).head(n)
    top_n_sums = (
        top_n_orgs
        .groupby(['italian_institution', 'year_block', 'country_name'])['org_citations']
        .sum().reset_index(name='top_n_citations')
    )
    cr_df = pd.merge(country_totals, top_n_sums,
                     on=['italian_institution', 'year_block', 'country_name'], how='left')
    cr_df['top_n_citations'] = cr_df['top_n_citations'].fillna(
        cr_df['total_country_citations']
    )
    cr_df[f'CR_{n}'] = (
        cr_df['top_n_citations'] / cr_df['total_country_citations']
    ) * 100

    return cr_df.sort_values(
        by=['italian_institution', 'year_block', 'total_country_citations'],
        ascending=[True, True, False]
    )

def load_temporal_data():
    master_df = load_all_temporal(dataset_type='organizations')

    if master_df.empty:
        print("No annualized org files found — check ANNUALIZED_BASE_PATH in data_utils.")
        return None, None, None

    if 'institution' in master_df.columns and 'italian_institution' not in master_df.columns:
        master_df = master_df.rename(columns={'institution': 'italian_institution'})

    return (
        master_df,
        calculate_concentration_metrics_temporal(master_df, 'incoming', 3),
        calculate_concentration_metrics_temporal(master_df, 'outgoing', 3),
    )


master_df, incoming_cr_df, outgoing_cr_df = load_temporal_data()


#### Optional: Export cleaned datasets

The cell below is intentionally commented out — it is not part of the regular analysis pipeline, and does not need to run for the rest of the notebook to work.
- `export_cleaned_csvs_temporal()` generates cleaned versions of the original datasets after normalization and aggregation.

In [ ]:
# Export cleaned temporal CSVs (Both Orgs and Countries!)
# export_cleaned_csvs_temporal(
#     output_dir=VISUALIZATIONS_PATH,
# )



Temporal Export complete: 120 files processed successfully, 0 files skipped.


## 2.2 Scatter Plot: Volume vs. Concentration Over Time

The following cell displays the same bubble scatter plot from Part I, but animated across the five time blocks. Each frame is one 5-year period. Use the ▶ play button or drag the slider to move through time. The Institution and Direction dropdowns control what is shown.

**What to look for as the animation plays:**

1. **Rightward drift** — bubbles moving right signal citation volume growth. Nearly all countries drift right; the speed separates rapidly emerging partners from stable ones.
2. **Downward drift** — bubbles moving down mean a broader set of the country's institutions is engaging. This is the structural signature of system-wide integration replacing bilateral partnerships.
3. **Crossing the 50% line** — the moment a country drops below the dashed threshold marks its transition from a niche to a systemic relationship.
4. **Outlier persistence** — countries that stay above CR₃ = 50% across multiple periods maintain structural dependencies regardless of volume growth.
5. **The 2021–2025 pullback** — a slight leftward movement in the final block is visible across institutions. This is a citation lag artefact: papers published in 2023–2025 have had less time to accumulate citations, so the final block undercounts volume relative to earlier ones.

In [6]:
# VISUALIZATION ENGINE
def plot_single_inst_animated_v2(inst_name, cr_df, direction, n=3):
    # Filter to this institution
    plot_df = cr_df[cr_df['italian_institution'] == inst_name].copy()
    plot_df = plot_df.sort_values('year_block')
 
    # Keep the top-25 countries by peak volume
    top_countries = (
        plot_df.groupby('country_name')['total_country_citations']
               .max()
               .nlargest(25)
               .index
    )
    plot_df = plot_df[plot_df['country_name'].isin(top_countries)].copy()
 
    # Decide which countries get permanent text labels
    # Only the top-10 by peak volume; others appear in hover only.
    top10_label_countries = (
        plot_df.groupby('country_name')['total_country_citations']
               .max()
               .nlargest(10)
               .index
               .tolist()
    )
    plot_df['label_text'] = plot_df['country_name'].where(
        plot_df['country_name'].isin(top10_label_countries), other=''
    )
 
    min_x = max(500,  plot_df['total_country_citations'].min() * 0.4)
    max_x =           plot_df['total_country_citations'].max() * 2.0
 
    cr_col = f'CR_{n}'
    direction_label = 'Incoming' if direction == 'incoming' else 'Outgoing'
 
    base_color = DIR_COLORS[direction]

    fig = px.scatter(
        plot_df,
        x='total_country_citations',
        y=cr_col,
        size='total_country_citations',
        color='country_name',
        text='label_text',
        log_x=True,
        template='plotly_white',
        height=620,
        size_max=55,
        color_discrete_sequence=px.colors.qualitative.Pastel,
        animation_frame='year_block',
        animation_group='country_name',
        range_x=[min_x, max_x],
        range_y=[0, 110],
        custom_data=['country_name', 'year_block', 'total_country_citations', cr_col],
        title=(
            f'<b>Citation Concentration vs. Volume</b>  ·  '
            f'{INST_LABELS.get(inst_name, inst_name)}  ·  {direction_label} Citations'
        ),
    )
 
    # Hover template
    fig.update_traces(
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Period: %{customdata[1]}<br>'
            'Total citations: %{customdata[2]:,.0f}<br>'
            f'CR₃ (top-3 share): %{{customdata[3]:.1f}}%'
            '<extra></extra>'
        ),
        textposition='top center',
        marker=dict(opacity=0.85, sizemode='area'),
    )
 
    # 50% line — "concentrated" threshold
    fig.add_hline(y=50, line_dash='dash', line_color=base_color, opacity=0.55)
    fig.add_annotation(
        xref='paper', x=0.01,
        yref='y',     y=50,
        text='Concentrated (CR₃ = 50%)',
        showarrow=False,
        xanchor='left',
        yanchor='bottom',
        font=dict(size=11, color=base_color),
        bgcolor='rgba(255,255,255,0.7)',
    )

    # 25% line — "moderate" threshold
    fig.add_hline(y=25, line_dash='dot', line_color=base_color, opacity=0.35)
    fig.add_annotation(
        xref='paper', x=0.01,
        yref='y',     y=25,
        text='Moderate (CR₃ = 25%)',
        showarrow=False,
        xanchor='left',
        yanchor='bottom',
        font=dict(size=11, color=base_color),
        bgcolor='rgba(255,255,255,0.7)',
    )
 
    # Axis formatting
    fig.update_xaxes(
        title_text='Total citation volume (log scale)',
        title_font_size=13,
        tickfont_size=11,
        tickvals=[1_000, 2_000, 5_000, 10_000, 20_000, 50_000,
                  100_000, 200_000, 500_000, 1_000_000, 2_000_000],
        ticktext=['1k', '2k', '5k', '10k', '20k', '50k',
                  '100k', '200k', '500k', '1M', '2M'],
        showgrid=True, gridcolor='#ebebeb',
    )
    fig.update_yaxes(
        title_text='CR₃ — share of citations from top-3 organisations (%)',
        title_font_size=13,
        tickfont_size=11,
        ticksuffix='%',
        showgrid=True, gridcolor='#ebebeb',
        zeroline=False,
    )
 
    # Layout polish
    fig.update_layout(
        title_font_size=15,
        title_x=0.0,
        legend=dict(
            title_text='',
            x=1.01, y=1,
            xanchor='left', yanchor='top',
            font_size=11,
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ddd',
            borderwidth=1,
        ),
        margin=dict(l=70, r=200, t=70, b=60),
        transition={'duration': 400},
        sliders=[{
            'currentvalue': {
                'prefix': 'Period: ',
                'font': {'size': 13},
            },
        }],
    )
 
    return fig

# INTERACTIVE DASHBOARD
if incoming_cr_df is not None and not incoming_cr_df.empty:

    out_anim = widgets.Output()

    inst_anim_dd = widgets.Dropdown(
        options=[(INST_LABELS[i], i) for i in INSTITUTIONS],
        value='UNIBO',
        description='Institution:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px'),
    )
    dir_anim_dd = widgets.Dropdown(
        options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
        value='incoming',
        description='Direction:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='230px'),
    )

    anim_controls = widgets.HBox(
        [inst_anim_dd, dir_anim_dd],
        layout=widgets.Layout(gap='20px', margin='0 0 15px 0'),
    )

    def render_animated(inst_name, direction):
        cr_data = calculate_concentration_metrics_temporal(master_df, direction, 3)
        with out_anim:
            clear_output(wait=True)
            if not cr_data.empty:
                plot_single_inst_animated_v2(inst_name, cr_data, direction, 3).show()

    def on_anim_change(change):
        render_animated(inst_anim_dd.value, dir_anim_dd.value)

    inst_anim_dd.observe(on_anim_change, names='value')
    dir_anim_dd.observe(on_anim_change, names='value')

    display(anim_controls, out_anim)
    render_animated('UNIBO', 'incoming')

else:
    print("No data loaded — check BASE_PATH and file names.")

Output()

## 2.3 CR₃ Snapshot Comparison: 2001–2005 vs 2021–2025

The animated scatter plot reveals trajectories, but comparing specific values across
institutions and time periods is difficult to do by eye. This table distils the
animation into a single static snapshot: CR₃ values for the 15 most-cited partner
countries, contrasted between the first (2001–2005) and most recent (2021–2025)
five-year block.

**How to read it:**

- Each column pair under an institution shows the CR₃ for that direction in 2001–2005
  (left) and 2021–2025 (right). A falling value signals structural diversification
  over the 25-year period; a rising value would indicate increasing dependency.
- The colour scale runs from yellow (low concentration, dispersed) to red (high
  concentration, hub-and-spoke). Values are directly comparable across institutions
  and directions.
- A `—` cell means the country had insufficient citation volume in that period to
  compute a meaningful CR₃.

The table is also exported as a high-resolution image for use in the paper and
presentation (see cell below).

In [7]:
def build_cr_comparison_table(master_df, n=3):
    """
    Build a CR comparison table: 2001-2005 vs 2021-2025
    Rows: top-10 countries by peak volume across all institutions & directions
    Columns: multi-level — institution > direction > period
    """
    # ── 1. Compute CR metrics for both directions ──
    incoming_cr = calculate_concentration_metrics_temporal(master_df, 'incoming', n)
    outgoing_cr = calculate_concentration_metrics_temporal(master_df, 'outgoing', n)
    incoming_cr['direction'] = 'incoming'
    outgoing_cr['direction'] = 'outgoing'
    cr_df = pd.concat([incoming_cr, outgoing_cr], ignore_index=True)

    cr_col = f'CR_{n}'

    # ── 2. Filter to the two periods of interest ──
    periods = ['2001-2005', '2021-2025']
    cr_df = cr_df[cr_df['year_block'].isin(periods)]

    # ── 3. Top-15 countries by peak total_country_citations across all institutions & directions ──
    top15 = (
        cr_df.groupby('country_name')['total_country_citations']
             .max()
             .nlargest(15)
             .index
             .tolist()
    )
    cr_df = cr_df[cr_df['country_name'].isin(top15)]

    # ── 4. Pivot into wide format ──
    pivot = cr_df.pivot_table(
        index='country_name',
        columns=['italian_institution', 'direction', 'year_block'],
        values=cr_col,
        aggfunc='first'
    )

    # ── 5. Reorder columns: institution > direction > period ──
    pivot = pivot.reindex(columns=pd.MultiIndex.from_product([
        INSTITUTIONS,
        ['incoming', 'outgoing'],
        periods,
    ]))

    # ── 6. Reorder rows by peak volume ──
    pivot = pivot.reindex(
        cr_df.groupby('country_name')['total_country_citations']
             .max()
             .nlargest(15)
             .index
    )

    return pivot


def display_cr_comparison_table(master_df, n=3):
    pivot = build_cr_comparison_table(master_df, n)

    def fmt(val):
        if pd.isna(val):
            return '—'
        return f'{val:.1f}%'

    styled = (
        pivot
        .style
        .format(fmt)
        .set_caption(
            f'Citation Concentration vs Volume — '
            f'2001–2005 vs 2021–2025'
        )
        .background_gradient(
            cmap='YlOrRd',
            vmin=0, vmax=100,
            axis=None,
        )
        .set_table_styles([
            # Caption
            {'selector': 'caption',
             'props': [('font-size', '14px'), ('font-weight', 'bold'),
                       ('text-align', 'left'), ('margin-bottom', '8px'),
                       ('color', '#111')]},
            # All header cells
            {'selector': 'th',
             'props': [('font-size', '11px'), ('text-align', 'center'),
                       ('padding', '4px 8px'), ('background-color', '#f0f0f0'),
                       ('color', '#111'), ('border', '1px solid #ddd')]},
            # Row index cells
            {'selector': 'th.row_heading',
             'props': [('text-align', 'left'), ('font-weight', 'bold'),
                       ('color', '#111'), ('background-color', '#f0f0f0'),
                       ('min-width', '120px')]},
            # Data cells
            {'selector': 'td',
             'props': [('font-size', '11px'), ('text-align', 'center'),
                       ('padding', '4px 8px'), ('border', '1px solid #eee')]},
            # Corner cell
            {'selector': 'th.blank',
             'props': [('background-color', '#f0f0f0')]},
        ])
    )

    display(styled)
    return pivot


# ── Run ──
pivot = display_cr_comparison_table(master_df, n=3)

In [8]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

def export_cr_table_image(pivot, path, n=3):
    
    # ── Format values ──
    formatted = pivot.applymap(lambda x: f'{x:.1f}%' if pd.notna(x) else '—')
    
    # ── Color mapping (same YlOrRd, 0–100 range) ──
    cmap = plt.cm.YlOrRd
    norm = mcolors.Normalize(vmin=0, vmax=100)
    
    def cell_color(val):
        if pd.isna(val):
            return (0.95, 0.95, 0.95, 1)  # light grey for missing
        return cmap(norm(val))
    
    cell_colors = [[cell_color(v) for v in row] for row in pivot.values]
    
    # ── Build column labels (3 levels: institution / direction / period) ──
    col_labels = [f'{i}\n{d}\n{p}' for i, d, p in pivot.columns]
    row_labels = pivot.index.tolist()
    
    # ── Figure size: scale with number of columns ──
    n_rows = len(row_labels)
    n_cols = len(col_labels)
    fig_width = max(20, n_cols * 0.9)
    fig_height = max(4, n_rows * 0.5 + 1.5)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    
    table = ax.table(
        cellText=formatted.values,
        rowLabels=row_labels,
        colLabels=col_labels,
        cellColours=cell_colors,
        cellLoc='center',
        loc='upper center',   
        bbox=[0, 0, 1, 1],   
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 2)

    # Move table up to close the gap
    table_bbox = table.get_window_extent(fig.canvas.get_renderer())
    ax.set_position([0.05, 0.02, 0.9, 0.88])  # [left, bottom, width, height]

    plt.subplots_adjust(top=0.93, bottom=0.02, left=0.05, right=0.97)
    
    plt.suptitle(
        f'Citation Concentration vs Volume — 2001–2005 vs 2021–2025',
        fontsize=12, fontweight='bold'
    )
    
    plt.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    
    # ── Style header and index cells ──
    for (row, col), cell in table.get_celld().items():
        if row == 0 or col == -1:  # header row or index column
            cell.set_facecolor('#f0f0f0')
            cell.set_text_props(fontweight='bold', color='#111')
        cell.set_edgecolor('#ddd')
    
    # ── Colorbar ──
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                        fraction=0.01, pad=0.01, shrink=0.6)
    cbar.set_label('CR₃ (%)', fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    
    plt.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved to {path}")


# Uncomment this part of code to export the png image of the table
# pivot = build_cr_comparison_table(master_df, n=3)
# export_cr_table_image(
#     pivot,
#     BASE_PATH.parent / "visualizations" / "tables" / "cr_comparison.png"
# )

## 2.4 Findings — 25-Year Network Evolution

### The dominant macro-trend: right and down

Across all six institutions and both directions, the movement from 2001–2005 to 2021–2025 is rightward (volume growth) combined with downward (falling CR₃). This is a structural property of citation network growth: as the total number of citations from a country increases, the probability that any single organisation maintains a dominant share decreases mechanically, because new citations distribute across an ever-broader institutional base.

### Trajectories of Key Global Partners

**The Stable Anchor — United States.**  
The US occupies the extreme bottom-right from the earliest time block onward. Its CR₃ remains below 10% throughout, confirming that even in 2001–2005 the American science system was too large and diverse to concentrate in a few institutions. Its bubble grows rightward every period; its structural position barely changes.

**The Rapid Emergence — China.**  
In 2001–2005, China appears as a mid-tier partner with moderate volume and a CR₃ near 20–30%. By 2021–2025 it has accelerated into the highest-volume tier alongside the US and major European partners, while its CR₃ has dropped steeply. This arc tracks the structural maturation of China's research system: from a handful of early-adopter institutions (leading universities engaging internationally before the system-wide expansion) to a broad national network.

**The European Core — UK, Germany, France, Spain.**  
These partners move as a coherent group: stable, low-concentration profiles (CR₃ < 20%) drifting steadily rightward. They never spend significant time above the 50% threshold. This reflects deep, pre-existing integration within the European Research Area rather than the emergence of new bilateral ties.

**India's Structural Transformation.**  
Among all partners, India shows the steepest concentration *drop*. In 2001–2005 its CR₃ hovers near 60–65%; by 2021–2025 it falls below 15%. This tracks the rapid expansion and diversification of India's university research base over the same period.

**Persistent Concentrations — Small Northern European Countries.**  
Finland, Denmark, and Sweden remain structurally concentrated relative to their volume across most of the 25 years. Their citation relationships with Italian institutions appear to be driven by specific bilateral partnerships rather than system-wide integration. This may reflect disciplinary specialisation.

### Directional symmetry

Switching direction from incoming to outgoing produces nearly identical charts for every institution. The country ranking, cluster geometry, and individual CR₃ values are all preserved. The only systematic difference is a slight compression of the x-axis — outgoing citation counts are typically 10–20% lower than incoming — reflecting the general pattern that these Italian institutions receive more citations than they generate relative to their largest partners.

---
# Part III - Country and Organisation Citation Structure

## 3.1 Sunburst Chart

While the scatter plot and animated chart answer *how concentrated* each country's citation relationship is, they do not reveal *which organisations* are responsible. This sunburst chart completes the picture by showing the full organisational composition of every partner country's citation contribution.

The chart operates at two levels simultaneously:

- **Inner ring** — the top-20 partner countries for the selected institution and direction, each wedge sized by that country's total citation volume. The relative sizes reflect the same volume differences visible on the scatter plot's x-axis.
- **Outer ring** — the top-50 contributing organisations within each country, revealed by clicking a country wedge. Each organisation block is sized proportionally to its citation count. An *Others* slice absorbs any citations beyond the top-50, ensuring country wedge sizes always reflect the true full total.

**How to navigate:**
- Select an **institution** and **direction** with the two dropdowns — all 12 combinations are pre-computed for instant switching.
- Click any **country wedge** to drill down into its organisations.
- Click the **centre disc** to zoom back out to the country ring.
- Hover over any wedge for exact citation counts and percentage shares.

A country with a large inner wedge whose outer ring fans into many small, similarly-sized organisation slices is broadly integrated — no single institution dominates. A country whose outer ring is dominated by one or two large slices, by contrast, has a structurally concentrated relationship regardless of its raw volume.

In [9]:
# Sunburst: Country → Organisation by Direction
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

TOP_COUNTRIES = 20
TOP_ORGS      = 50
colors_20 = [
    '#4D1343', '#d62728', '#843c39', '#E8743B', '#e6550d',
    '#BE8B20', '#F4C430', '#B7990D', '#FFE94D', '#637939',
    '#A3E635', '#2ca02c', '#97DFFC', '#1f77b4', '#aec7e8',
    '#4361EE', '#5254a3', '#9467bd', '#5B2C6F', '#320E3B',
]

# Colour helpers
def _to_rgb(color):
    import re
    color = str(color).strip()
    if color.startswith('rgb'):
        nums = re.findall(r'[\d.]+', color)
        return int(float(nums[0])), int(float(nums[1])), int(float(nums[2]))
    h = color.lstrip('#')
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)


def _tint(color, factor):
    r, g, b = _to_rgb(color)
    return '#{:02x}{:02x}{:02x}'.format(
        int(r + (255 - r) * factor),
        int(g + (255 - g) * factor),
        int(b + (255 - b) * factor),
    )

def _weight_col(df):
    for c in ('counts', 'count', 'derived_counts'):
        if c in df.columns:
            return c
    df['derived_counts'] = 1
    return 'derived_counts'


def _wrap_label(name, max_chars=14):
    """Break a name into lines of at most max_chars using <br> at word boundaries."""
    words = name.split()
    lines, current, length = [], [], 0
    for word in words:
        if length + len(word) + (1 if current else 0) > max_chars and current:
            lines.append(' '.join(current))
            current, length = [word], len(word)
        else:
            current.append(word)
            length += len(word) + (1 if len(current) > 1 else 0)
    if current:
        lines.append(' '.join(current))
    return '<br>'.join(lines)


# Sunburst data builder
def make_sunburst_data(inst, direction):
    df_dir = agg_master[
        (agg_master['italian_institution'] == inst) &
        (agg_master['direction'] == direction)
    ].copy()
    if df_dir.empty:
        return None

    wcol = _weight_col(df_dir)
    label_inst = INST_LABELS.get(inst, inst)

    # Aggregate: country × org
    org_agg = (
        df_dir
        .groupby(['country_name', 'legal_name'])[wcol]
        .sum()
        .reset_index(name='citations')
    )

    # Country totals → top-N
    country_totals = (
        org_agg.groupby('country_name')['citations']
        .sum()
        .reset_index(name='country_total')
        .sort_values('country_total', ascending=False)
        .head(TOP_COUNTRIES)
    )
    grand_total     = country_totals['country_total'].sum()
    top_country_set = set(country_totals['country_name'])
    org_agg         = org_agg[org_agg['country_name'].isin(top_country_set)]

    # Top-N orgs per country
    top_orgs_idx = (
        org_agg
        .sort_values('citations', ascending=False)
        .groupby('country_name')
        .head(TOP_ORGS)
        [['country_name', 'legal_name']]
    )
    org_agg = org_agg.merge(top_orgs_idx, on=['country_name', 'legal_name'])

    ids, labels, parents, values, customdata, colors = \
        [], [], [], [], [], []

    # Root — wrap long institution names so they fit in the centre circle
    ids.append('root')
    labels.append(_wrap_label(label_inst, max_chars=14))
    parents.append('')
    values.append(0)
    customdata.append(f'<b>{label_inst}</b>')
    colors.append('rgba(0,0,0,0)')

    for ci, (_, crow) in enumerate(country_totals.iterrows()):
        country     = crow['country_name']
        ctotal      = int(crow['country_total'])
        country_pct = ctotal / grand_total * 100
        base_color  = colors_20[ci % len(colors_20)]   # ← by position, never grey
        org_color   = _tint(base_color, 0.30)

        # Country node
        ids.append(country)
        labels.append(country)
        parents.append('root')
        values.append(0)
        customdata.append(
            f'<b>{country}</b><br>'
            f'Citations: {ctotal:,}<br>'
            f'Share of {label_inst}: {country_pct:.1f}%'
        )
        colors.append(base_color)

        # Org nodes
        country_orgs = (
            org_agg[org_agg['country_name'] == country]
            .sort_values('citations', ascending=False)
        )
        for _, orow in country_orgs.iterrows():
            org    = orow['legal_name']
            ocites = int(orow['citations'])
            pct_c  = ocites / ctotal      * 100
            pct_g  = ocites / grand_total * 100

            ids.append(f'{country}__{org}')
            labels.append(org)
            parents.append(country)
            values.append(ocites)
            customdata.append(
                f'<b>{org}</b><br>'
                f'Country: {country}<br>'
                f'Citations: {ocites:,}<br>'
                f'% of country: {pct_c:.1f}%<br>'
                f'% of {label_inst}: {pct_g:.1f}%'
            )
            colors.append(org_color)

    return dict(ids=ids, labels=labels, parents=parents, values=values,
                customdata=customdata, colors=colors)


# Figure builder

def _build_figure(inst, direction):
    d           = make_sunburst_data(inst, direction)
    label_inst  = INST_LABELS.get(inst, inst)
    dir_label   = DIR_LABELS.get(direction, direction)
    dir_badge   = '▼ Incoming' if direction == 'incoming' else '▲ Outgoing'
    badge_color = DIR_COLORS.get(direction, '#555')
    fig         = go.Figure()
    if d is None:
        return fig

    fig.add_trace(go.Sunburst(
        ids=d['ids'],
        labels=d['labels'],
        parents=d['parents'],
        values=d['values'],
        customdata=d['customdata'],
        texttemplate='%{label}',
        hovertemplate='%{customdata}<extra></extra>',
        branchvalues='remainder',
        rotation=-90,
        sort=False,
        insidetextorientation='radial',
        leaf=dict(opacity=0.88),
        marker=dict(
            colors=d['colors'],
            line=dict(width=0.3, color='white'),
        ),
    ))

    fig.update_layout(
        font=dict(family='Inter, sans-serif', color='#333'),
        title=dict(
            text=(
                f'<b>Country → Organisation Citation Structure</b>  ·  {label_inst}'
                f'<br><sup>Top {TOP_COUNTRIES} countries · top {TOP_ORGS} orgs per country'
                f' · {dir_label} · click a country to reveal organisations</sup>'
            ),
            font=dict(family='Playfair Display, serif', size=18),
            x=0.0,
        ),
        margin=dict(l=10, r=10, t=90, b=20),
        height=650,
        template='plotly_white',
        showlegend=False,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )
    return fig


# Pre-compute all 12 figures
print("Pre-computing sunburst charts (6 institutions × 2 directions)…")
_FIGURE_CACHE = {}
if agg_master is not None and not agg_master.empty:
    for _inst in INSTITUTIONS:
        for _dir in ('incoming', 'outgoing'):
            _FIGURE_CACHE[(_inst, _dir)] = _build_figure(_inst, _dir)
    print(f"  Done — {len(_FIGURE_CACHE)} figures cached.")
else:
    print("  agg_master is empty — check load_all_available() in cell 1.2.")


# Widgets
if _FIGURE_CACHE:
    out_sb = widgets.Output()

    inst_sb_dd = widgets.Dropdown(
        options=[(INST_LABELS[i], i) for i in INSTITUTIONS],
        value='UNIBO',
        description='Institution:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px'),
    )
    dir_sb_dd = widgets.Dropdown(
        options=[('Incoming citations', 'incoming'), ('Outgoing citations', 'outgoing')],
        value='incoming',
        description='Direction:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='230px'),
    )
    sb_controls = widgets.HBox(
        [inst_sb_dd, dir_sb_dd],
        layout=widgets.Layout(gap='20px', margin='0 0 12px 0'),
    )

    def render_sb(inst, direction):
        with out_sb:
            clear_output(wait=True)
            _FIGURE_CACHE[(inst, direction)].show()

    def on_sb_change(change):
        render_sb(inst_sb_dd.value, dir_sb_dd.value)

    inst_sb_dd.observe(on_sb_change, names='value')
    dir_sb_dd.observe(on_sb_change, names='value')

    display(sb_controls, out_sb)
    render_sb('UNIBO', 'incoming')

from pathlib import Path

try:
    BASE = Path(__file__).resolve().parent
except NameError:
    BASE = Path.cwd()

OUT_DIR = (BASE / ".." / ".." / "website" / "visualizations" /  "sunburst").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_CSS = """
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&family=Playfair+Display:wght@600;700&display=swap" rel="stylesheet">
"""

# written = []
# for (inst, direction), fig_cached in _FIGURE_CACHE.items():
#     fname    = f"sunburst_{inst}_{direction}.html"
#     out_path = OUT_DIR / fname
#     with open(out_path, "w", encoding="utf-8") as f:
#         f.write(FONT_CSS + fig_cached.to_html(
#             full_html=False, include_plotlyjs=False, config={"responsive": True}))
#     written.append(fname)

# print(f"Wrote {len(written)} files to {OUT_DIR}:")
# for name in written:
#     print(f"  {name}")

Pre-computing sunburst charts (6 institutions × 2 directions)…
  Done — 12 figures cached.


Output()

## 3.2 Findings — Organisational Composition Patterns

The sunburst charts reveal the internal architecture of each country's citation relationship — the layer of analysis that CR₃ summarises but cannot show directly.

Exploring the chart, it is possible to answers that question directly. Each sunburst shows how the top-50 organizations contributing to a selected country’s citations are distributed: wide, even rings signal a fragmented network; a few large slices signal structural dependency on a handful of institutions.

The pattern is consistent and intuitive. **Large science systems — the United States, France, Italy — spread their citations across dozens of organizations**. No single institution dominates (except for CNRS); the relationship is systemic, not bilateral. These are the most resilient partnerships: removing any one organization would barely move the aggregate.

**Smaller countries tell the opposite story.** Their citation flows concentrate in one or two organizations. The relationship is valuable but fragile: it is a partnership with an institution, not with a science system.

This distinction matters for research strategy. High volume alone does not mean deep integration. A country with thousands of citations funnelled through one organization is structurally more exposed than a smaller partner whose citations are spread across ten.

### Dominant organisations mirror national science system structure

For all six Italian institutions, the outer ring of the largest partner countries (US, UK, Germany, France) fans into dozens of comparably-sized slices. No single foreign organisation monopolises even the largest country wedges. This is the visual counterpart of the near-zero CR₃ values observed in the scatter plot: the low concentration scores are not statistical artefacts — they reflect genuinely broad engagement across dozens of institutions within each major partner country.

### Small countries, single institutions

For the smaller partner countries — particularly in Northern Europe and parts of Asia — the outer ring typically resolves to one or two organisation slices filling the entire wedge. This confirms the concentration signal from the scatter plot: these are genuinely hub-and-spoke relationships driven by a single bilateral institutional partnership rather than system-wide engagement.